<a href="https://colab.research.google.com/github/2001lida/PythonLession2/blob/HW_9/%D0%97%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B59.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

import pandas as pd

df = pd.read_csv(
    "realty_data.csv",
    sep=",",
    engine="python",
    on_bad_lines="skip"
)

print(df.shape)

FEATURES = [
    "total_square",
    "rooms",
    "floor",
    "city",
    "district",
    "area",
    "object_type",
    "lat",
    "lon"
]

TARGET = "price"

df = df[FEATURES + [TARGET]].dropna()

X = df[FEATURES]
y = df[TARGET]

numeric_features = ["total_square", "rooms", "floor", "lat", "lon"]
categorical_features = ["city", "district", "area", "object_type"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model.fit(X_train, y_train)

joblib.dump(model, "model.pkl")

print("Модель обучена и сохранена")

(47594, 17)
Модель обучена и сохранена


In [5]:
import joblib
import pandas as pd

model = joblib.load("model.pkl")

sample = pd.DataFrame([{
    "total_square": 50,
    "rooms": 2,
    "floor": 5,
    "city": "Москва",
    "district": "Центральный",
    "area": "ЦАО",
    "object_type": "квартира",
    "lat": 55.75,
    "lon": 37.61
}])

print("Предсказание:", model.predict(sample)[0])

Предсказание: 14066719.28


In [9]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

model = joblib.load("model.pkl")

st.title("Прогноз стоимости недвижимости")

total_square = st.number_input("Площадь", 10.0, 500.0, 50.0)
rooms = st.number_input("Комнаты", 1, 10, 2)
floor = st.number_input("Этаж", 1, 50, 5)

city = st.text_input("Город", "Москва")
district = st.text_input("Район", "Центральный")
area = st.text_input("Округ", "ЦАО")
object_type = st.text_input("Тип объекта", "квартира")

lat = st.number_input("Широта", value=55.75)
lon = st.number_input("Долгота", value=37.61)

if st.button("Рассчитать цену"):
    data = pd.DataFrame([{
        "total_square": total_square,
        "rooms": rooms,
        "floor": floor,
        "city": city,
        "district": district,
        "area": area,
        "object_type": object_type,
        "lat": lat,
        "lon": lon
    }])

    prediction = model.predict(data)[0]

    st.success(f"Цена: {int(prediction):,} руб.")

Overwriting app.py


In [15]:
!pip install streamlit
!pip install pyngrok

In [17]:
from pyngrok import ngrok
import subprocess
import time
!ngrok config add-authtoken 3BzfCHUllnQZB3dvm9cUViUNVeI_2ws7aUrdsmhrAS1BHTMxQ

process = subprocess.Popen(["streamlit", "run", "app.py"])
time.sleep(5)

url = ngrok.connect(8501)
print("Открой:", url)

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Открой: NgrokTunnel: "https://unrepulsive-christinia-nonaesthetically.ngrok-free.dev" -> "http://localhost:8501"
